# 使用 logprobs 进行分类和问答评估

本 notebook 演示 Chat Completions API 中 `logprobs` 参数的用法。启用 `logprobs` 后，API 会返回每个输出 token 的对数概率，以及每个 token 位置上数量有限的最可能 token 及其对数概率。相关的请求参数如下：
* `logprobs`：是否返回输出 token 的对数概率。如果为 `true`，响应中将包含消息内容中每个 token 的对数概率。
* `top_logprobs`：一个介于 0 到 5 之间的整数，指定在每个 token 位置返回的最可能 token 的数量，每个 token 都附带其对数概率。使用此参数时必须将 `logprobs` 设置为 `true`。

输出 token 的对数概率表示在给定上下文的情况下，每个 token 出现在序列中的可能性。简单来说，logprob 就是 `log(p)`，其中 `p` = 基于上下文中前面的 token，某个 token 出现在特定位置的概率。关于 `logprobs` 的几个要点：
* 对数概率越高，表示该 token 在该上下文中出现的可能性越大。这让用户可以衡量模型对其输出的置信度，或探索模型考虑过的其他回复。
* Logprob 可以是任意负数或 `0.0`。`0.0` 对应 100% 的概率。
* 通过对序列中各 token 的 logprob 求和，我们可以计算该序列的联合对数概率。对该和取指数即可得到联合概率。这对于给模型输出打分和排序很有用。另一种常见做法是取句子中每个 token 的平均 logprob 来选择最佳生成结果。
* 我们可以检查分配给不同候选 token 的 `logprobs`，以了解模型认为哪些选项是合理的、哪些是不合理的。

虽然 `logprobs` 有非常广泛的用例，但本 notebook 将重点关注以下用途：

1. **分类任务：** 大语言模型擅长许多分类任务，但准确衡量模型对其输出的置信度可能颇具挑战。`logprobs` 为每个类别预测提供了一个关联的概率，让用户可以设置分类或置信度阈值。
2. **检索（问答）评估：** `logprobs` 可以辅助检索应用中的自我评估。在问答示例中，模型会输出一个人为设计的 `has_sufficient_context_for_answer` 布尔值，它可以作为答案是否包含在检索内容中的置信度分数。此类评估可以减少基于检索的幻觉并提高准确性。
3. **自动补全：** `logprobs` 可以帮助我们决定在用户输入时何时推荐词语。
4. **Token 高亮和输出字节：** 用户可以利用启用 `logprobs` 后自带的内置分词结果创建一个 token 高亮工具。`bytes` 字段还包含每个输出 token 的 UTF-8 字节值，这对于还原 emoji 和特殊字符特别有用。
5. **计算困惑度：** `logprobs` 可以帮助我们评估模型对某个结果的整体置信度，并比较不同提示所得结果的置信度。

## 0. 导入与工具函数

在运行本 notebook 之前，请设置 `OPENAI_API_KEY` 环境变量。

In [ ]:
from openai import OpenAI
from math import exp
import numpy as np
from IPython.display import display, HTML
import os

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "<your OpenAI API key if not set as env var>"))

In [31]:
def get_completion(
    messages: list[dict[str, str]],
    model: str = "gpt-4",
    max_tokens=500,
    temperature=0,
    stop=None,
    seed=123,
    tools=None,
    logprobs=None,  # whether to return log probabilities of the output tokens or not. If true, returns the log probabilities of each output token returned in the content of message..
    top_logprobs=None,
) -> str:
    params = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "stop": stop,
        "seed": seed,
        "logprobs": logprobs,
        "top_logprobs": top_logprobs,
    }
    if tools:
        params["tools"] = tools

    completion = client.chat.completions.create(**params)
    return completion

## 1. 使用 `logprobs` 评估分类任务的置信度

假设我们想创建一个系统，将新闻文章分类到一组预定义的类别中。没有 `logprobs` 时，我们也可以使用 Chat Completions 来完成这项任务，但很难评估模型做出分类时的确定程度。

现在，启用 `logprobs` 后，我们可以准确地看到模型对其预测有多大把握，这对于构建准确可信的分类器至关重要。例如，如果所选类别的对数概率很高，说明模型对其分类相当有信心；如果很低，则说明模型信心不足。当模型的分类结果与你的预期不符，或者模型的输出需要由人工审核或验证时，这一点尤其有用。

我们从一个提示开始，向模型提供四个类别：**Technology、Politics、Sports 和 Arts**。然后让模型仅根据标题将文章分类到这些类别中。

In [32]:
CLASSIFICATION_PROMPT = """You will be given a headline of a news article.
Classify the article into one of the following categories: Technology, Politics, Sports, and Art.
Return only the name of the category, and nothing else.
MAKE SURE your output is one of the four categories stated.
Article headline: {headline}"""


我们来看三个示例标题，首先从不带 `logprobs` 的标准 Chat Completions 输出开始

In [33]:
headlines = [
    "Tech Giant Unveils Latest Smartphone Model with Advanced Photo-Editing Features.",
    "Local Mayor Launches Initiative to Enhance Urban Public Transport.",
    "Tennis Champion Showcases Hidden Talents in Symphony Orchestra Debut",
]


In [69]:
for headline in headlines:
    print(f"\nHeadline: {headline}")
    API_RESPONSE = get_completion(
        [{"role": "user", "content": CLASSIFICATION_PROMPT.format(headline=headline)}],
        model="gpt-4o",
    )
    print(f"Category: {API_RESPONSE.choices[0].message.content}\n")


Headline: Tech Giant Unveils Latest Smartphone Model with Advanced Photo-Editing Features.
Category: Technology


Headline: Local Mayor Launches Initiative to Enhance Urban Public Transport.
Category: Politics


Headline: Tennis Champion Showcases Hidden Talents in Symphony Orchestra Debut
Category: Art



这里我们可以看到每个标题所选的类别。然而，我们无法了解模型对其预测的置信度。让我们在启用 `logprobs` 并将 `top_logprobs` 设置为 2 的情况下重新运行同一个提示，这样会显示每个位置上最可能的两个输出 token。我们还可以将每个对数概率转换为更易解读的 0–100% 线性概率。

In [57]:
for headline in headlines:
    print(f"\nHeadline: {headline}")
    API_RESPONSE = get_completion(
        [{"role": "user", "content": CLASSIFICATION_PROMPT.format(headline=headline)}],
        model="gpt-4o-mini",
        logprobs=True,
        top_logprobs=2,
    )
    top_two_logprobs = API_RESPONSE.choices[0].logprobs.content[0].top_logprobs
    html_content = ""
    for i, logprob in enumerate(top_two_logprobs, start=1):
        html_content += (
            f"<span style='color: cyan'>Output token {i}:</span> {logprob.token}, "
            f"<span style='color: darkorange'>logprobs:</span> {logprob.logprob}, "
            f"<span style='color: magenta'>linear probability:</span> {np.round(np.exp(logprob.logprob)*100,2)}%<br>"
        )
    display(HTML(html_content))
    print("\n")


Headline: Tech Giant Unveils Latest Smartphone Model with Advanced Photo-Editing Features.





Headline: Local Mayor Launches Initiative to Enhance Urban Public Transport.





Headline: Tennis Champion Showcases Hidden Talents in Symphony Orchestra Debut


对于前两个标题，正如预期的那样，gpt-4o-mini 对其分类有 100% 的信心，因为其内容显然分别聚焦于科技和政治。然而，第三个标题结合了体育和艺术相关的主题，因此置信度略低，为 97%，但仍然表现出对其分类的高度确定。

`logprobs` 对分类任务非常有用。它们让我们可以设置置信度阈值，或者在所选输出的对数概率不够高时输出多个可能的 token。例如，在构建为文章打标签的推荐引擎时，我们可以自动分类超过某一阈值的标题，并将不太确定的标题送去人工审核。

## 2. 通过检索置信度评分减少幻觉

为了减少幻觉并改进基于 RAG 的问答系统，我们可以使用 `logprobs` 来评估模型是否认为自己拥有足够的检索上下文。

假设我们已经使用 RAG 构建了一个用于问答的检索系统，但一直苦于问题答案中出现的幻觉。*注意：* 本示例将使用一篇硬编码的文章，但关于使用 RAG 进行问答的教程，请参阅 cookbook 中的其他条目。

In [36]:
# Article retrieved
ada_lovelace_article = """Augusta Ada King, Countess of Lovelace (née Byron; 10 December 1815 – 27 November 1852) was an English mathematician and writer, chiefly known for her work on Charles Babbage's proposed mechanical general-purpose computer, the Analytical Engine. She was the first to recognise that the machine had applications beyond pure calculation.
Ada Byron was the only legitimate child of poet Lord Byron and reformer Lady Byron. All Lovelace's half-siblings, Lord Byron's other children, were born out of wedlock to other women. Byron separated from his wife a month after Ada was born and left England forever. He died in Greece when Ada was eight. Her mother was anxious about her upbringing and promoted Ada's interest in mathematics and logic in an effort to prevent her from developing her father's perceived insanity. Despite this, Ada remained interested in him, naming her two sons Byron and Gordon. Upon her death, she was buried next to him at her request. Although often ill in her childhood, Ada pursued her studies assiduously. She married William King in 1835. King was made Earl of Lovelace in 1838, Ada thereby becoming Countess of Lovelace.
Her educational and social exploits brought her into contact with scientists such as Andrew Crosse, Charles Babbage, Sir David Brewster, Charles Wheatstone, Michael Faraday, and the author Charles Dickens, contacts which she used to further her education. Ada described her approach as "poetical science" and herself as an "Analyst (& Metaphysician)".
When she was eighteen, her mathematical talents led her to a long working relationship and friendship with fellow British mathematician Charles Babbage, who is known as "the father of computers". She was in particular interested in Babbage's work on the Analytical Engine. Lovelace first met him in June 1833, through their mutual friend, and her private tutor, Mary Somerville.
Between 1842 and 1843, Ada translated an article by the military engineer Luigi Menabrea (later Prime Minister of Italy) about the Analytical Engine, supplementing it with an elaborate set of seven notes, simply called "Notes".
Lovelace's notes are important in the early history of computers, especially since the seventh one contained what many consider to be the first computer program—that is, an algorithm designed to be carried out by a machine. Other historians reject this perspective and point out that Babbage's personal notes from the years 1836/1837 contain the first programs for the engine. She also developed a vision of the capability of computers to go beyond mere calculating or number-crunching, while many others, including Babbage himself, focused only on those capabilities. Her mindset of "poetical science" led her to ask questions about the Analytical Engine (as shown in her notes) examining how individuals and society relate to technology as a collaborative tool.
"""

# Questions that can be easily answered given the article
easy_questions = [
    "What nationality was Ada Lovelace?",
    "What was an important finding from Lovelace's seventh note?",
]

# Questions that are not fully covered in the article
medium_questions = [
    "Did Lovelace collaborate with Charles Dickens",
    "What concepts did Lovelace build with Charles Babbage",
]

现在，我们可以让模型评估所提供的上下文是否足以回答某个问题。具体来说，我们会让模型输出一个布尔值 `has_sufficient_context_for_answer`。然后，我们可以评估 `logprobs`，看看模型对这一分类有多大把握。

In [61]:
PROMPT = """You retrieved this article: {article}. The question is: {question}.
Before even answering the question, consider whether you have sufficient information in the article to answer the question fully.
Your output should JUST be the boolean true or false, of if you have sufficient information in the article to answer the question.
Respond with just one word, the boolean true or false. You must output the word 'True', or the word 'False', nothing else.
"""


In [65]:
html_output = ""
html_output += "Questions clearly answered in article"

for question in easy_questions:
    API_RESPONSE = get_completion(
        [
            {
                "role": "user",
                "content": PROMPT.format(
                    article=ada_lovelace_article, question=question
                ),
            }
        ],
        model="gpt-4o-mini",
        logprobs=True,
    )
    html_output += f'<p style="color:green">Question: {question}</p>'
    for logprob in API_RESPONSE.choices[0].logprobs.content:
        html_output += f'<p style="color:cyan">has_sufficient_context_for_answer: {logprob.token}, <span style="color:darkorange">logprobs: {logprob.logprob}, <span style="color:magenta">linear probability: {np.round(np.exp(logprob.logprob)*100,2)}%</span></p>'

html_output += "Questions only partially covered in the article"

for question in medium_questions:
    API_RESPONSE = get_completion(
        [
            {
                "role": "user",
                "content": PROMPT.format(
                    article=ada_lovelace_article, question=question
                ),
            }
        ],
        model="gpt-4o",
        logprobs=True,
        top_logprobs=3,
    )
    html_output += f'<p style="color:green">Question: {question}</p>'
    for logprob in API_RESPONSE.choices[0].logprobs.content:
        html_output += f'<p style="color:cyan">has_sufficient_context_for_answer: {logprob.token}, <span style="color:darkorange">logprobs: {logprob.logprob}, <span style="color:magenta">linear probability: {np.round(np.exp(logprob.logprob)*100,2)}%</span></p>'

display(HTML(html_output))

对于前两个问题，模型以接近 100% 的置信度返回 `True`，认为文章包含足够的上下文。

对于部分覆盖的问题，模型同样非常自信：对于 Charles Dickens 的问题，它以 99.14% 的概率返回 `False`；对于 Charles Babbage 的问题，它以 99.59% 的概率返回 `True`。这表明置信度衡量的是对分类的确定程度，而不是分类是否正确。

这种自我评估可以作为一道护栏，但在使用它来限制回答或提出后续问题之前，请先用针对具体任务的评估来验证其分类和阈值。有关相关方法的示例，请参阅 [How we cut the rate of GPT hallucinations](https://jfan001.medium.com/how-we-cut-the-rate-of-gpt-hallucinations-from-20-to-less-than-2-f3bfcc10e4ec)。

## 3. 自动补全

`logprobs` 的另一个用例是自动补全。在不构建完整自动补全系统的前提下，我们来演示 `logprobs` 如何帮助我们决定在用户输入时何时推荐词语。

首先，我们构造一个示例句子：`"My least favorite TV show is Breaking Bad."` 假设我们希望在输入这个句子时动态推荐下一个单词或 token，但*仅*在模型相当确定下一个词是什么时才推荐。为了演示这一点，我们将句子拆分为依次递进的片段。

In [39]:
sentence_list = [
    "My",
    "My least",
    "My least favorite",
    "My least favorite TV",
    "My least favorite TV show",
    "My least favorite TV show is",
    "My least favorite TV show is Breaking Bad",
]

现在，我们可以让 `gpt-4o-mini` 根据所给的任意上下文充当自动补全引擎。我们可以启用 `logprobs`，查看模型对其预测有多大把握。

In [ ]:
high_prob_completions = {}
low_prob_completions = {}
html_output = ""

for sentence in sentence_list:
    PROMPT = """Complete this sentence. You are acting as auto-complete. Simply complete the sentence to the best of your ability, make sure it is just ONE sentence: {sentence}"""
    API_RESPONSE = get_completion(
        [{"role": "user", "content": PROMPT.format(sentence=sentence)}],
        model="gpt-4o-mini",
        logprobs=True,
        top_logprobs=3,
    )
    html_output += f'<p>Sentence: {sentence}</p>'
    first_token = True
    for token in API_RESPONSE.choices[0].logprobs.content[0].top_logprobs:
        html_output += f'<p style="color:cyan">Predicted next token: {token.token}, <span style="color:darkorange">logprobs: {token.logprob}, <span style="color:magenta">linear probability: {np.round(np.exp(token.logprob)*100,2)}%</span></p>'
        if first_token:
            if np.exp(token.logprob) > 0.95:
                high_prob_completions[sentence] = token.token
            if np.exp(token.logprob) < 0.60:
                low_prob_completions[sentence] = token.token
        first_token = False
    html_output += "<br>"

display(HTML(html_output))

我们来看看高置信度的自动补全：

In [48]:
high_prob_completions


{'My least favorite TV show is Breaking Bad': 'because'}

这些看起来很合理！我们可以对这些建议充满信心。在写下 'My least favorite TV' 之后，你很可能想写 'show'！现在我们来看看模型不太有把握的自动补全建议：

In [49]:
low_prob_completions


{'My least favorite TV': 'show', 'My least favorite TV show': 'is'}

这些同样合乎逻辑。仅凭前缀 'my least favorite'，很难确定用户要说什么，而作者最喜欢的电视剧是什么，更是谁也猜不准。<br><br>
因此，借助 `gpt-4o-mini` 和 `logprobs`，我们就能搭建一个动态自动补全引擎的雏形！

## 4. 高亮工具与 bytes 参数

我们来使用 `logprobs` 创建一个简单的 token 高亮工具，并使用 `bytes` 字段。首先，我们可以创建一个函数来统计并高亮每个 token。虽然这并未用到对数概率，但它使用了启用 `logprobs` 后自带的内置分词结果。

In [66]:
PROMPT = """What's the longest word in the English language?"""

API_RESPONSE = get_completion(
    [{"role": "user", "content": PROMPT}], model="gpt-4o", logprobs=True, top_logprobs=5
)


def highlight_text(api_response):
    colors = [
        "#FF00FF",  # Magenta
        "#008000",  # Green
        "#FF8C00",  # Dark Orange
        "#FF0000",  # Red
        "#0000FF",  # Blue
    ]
    tokens = api_response.choices[0].logprobs.content

    color_idx = 0  # Initialize color index
    html_output = ""  # Initialize HTML output
    for t in tokens:
        token_str = bytes(t.bytes).decode("utf-8")  # Decode bytes to string

        # Add colored token to HTML output
        html_output += f"<span style='color: {colors[color_idx]}'>{token_str}</span>"

        # Move to the next color
        color_idx = (color_idx + 1) % len(colors)
    display(HTML(html_output))  # Display HTML output
    print(f"Total number of tokens: {len(tokens)}")

In [67]:
highlight_text(API_RESPONSE)


Total number of tokens: 95


接下来，我们使用 `bytes` 字段重建一个句子。启用 `logprobs` 后，响应中会包含每个 token 及其文本的 UTF-8 字节值。当 token 包含 emoji 或其他特殊字符时，这些字节值非常有用。

In [68]:
PROMPT = """Output the blue heart emoji and its name."""
API_RESPONSE = get_completion(
    [{"role": "user", "content": PROMPT}], model="gpt-4o", logprobs=True
)

aggregated_bytes = []
joint_logprob = 0.0

# Iterate over tokens, aggregate bytes and calculate joint logprob
for token in API_RESPONSE.choices[0].logprobs.content:
    print("Token:", token.token)
    print("Log prob:", token.logprob)
    print("Linear prob:", np.round(exp(token.logprob) * 100, 2), "%")
    print("Bytes:", token.bytes, "\n")
    aggregated_bytes += token.bytes
    joint_logprob += token.logprob

# Decode the aggregated bytes to text
aggregated_text = bytes(aggregated_bytes).decode("utf-8")

# Assert that the decoded text is the same as the message content
assert API_RESPONSE.choices[0].message.content == aggregated_text

# Print the results
print("Bytes array:", aggregated_bytes)
print(f"Decoded bytes: {aggregated_text}")
print("Joint prob:", np.round(exp(joint_logprob) * 100, 2), "%")

Token: Here
Log prob: -0.054242473
Linear prob: 94.72 %
Bytes: [72, 101, 114, 101] 

Token:  is
Log prob: -0.0044352207
Linear prob: 99.56 %
Bytes: [32, 105, 115] 

Token:  the
Log prob: -2.1008714e-06
Linear prob: 100.0 %
Bytes: [32, 116, 104, 101] 

Token:  blue
Log prob: -0.0013290489
Linear prob: 99.87 %
Bytes: [32, 98, 108, 117, 101] 

Token:  heart
Log prob: 0.0
Linear prob: 100.0 %
Bytes: [32, 104, 101, 97, 114, 116] 

Token:  emoji
Log prob: 0.0
Linear prob: 100.0 %
Bytes: [32, 101, 109, 111, 106, 105] 

Token:  and
Log prob: -0.038287632
Linear prob: 96.24 %
Bytes: [32, 97, 110, 100] 

Token:  its
Log prob: 0.0
Linear prob: 100.0 %
Bytes: [32, 105, 116, 115] 

Token:  name
Log prob: -1.569009e-05
Linear prob: 100.0 %
Bytes: [32, 110, 97, 109, 101] 

Token: :


Log prob: -0.11313002
Linear prob: 89.3 %
Bytes: [58, 10, 10] 

Token: \xf0\x9f\x92
Log prob: -0.09048584
Linear prob: 91.35 %
Bytes: [240, 159, 146] 

Token: \x99
Log prob: 0.0
Linear prob: 100.0 %
Bytes: [153] 

Token:

这里，emoji 被拆分到两个 token 中，它们的字节值组合起来构成其 UTF-8 序列。通过将所有 token 的字节追加到一个数组中，我们可以将数组解码为完整的句子，并断言它与补全消息一致。

我们还可以通过对 token 的对数概率求和并对结果取指数，来计算补全的联合概率。这表示对于该提示而言，完整补全出现的可能性有多大。由于这个提示具有明确的指向性，要求给出一个特定的 emoji 及其名称，因此联合概率很高。约束较少的提示可能会产生较低的联合概率，这在提示工程中可以作为一个有用的信号。

## 5. 计算困惑度

在评估模型对某个结果的置信度时，计算困惑度（perplexity）会很有用，它是衡量不确定性的指标。困惑度可以通过对 logprob 平均值的相反数取指数来计算。一般来说，困惑度越高表示结果越不确定，困惑度越低表示结果越有把握。因此，困惑度既可以用来评估单次模型运行的结果，也可以用来比较不同模型运行之间结果的相对置信度。虽然高置信度并不能保证结果的准确性，但它可以作为一个有用的信号，与其他评估指标结合使用，帮助你更好地理解提示的行为。

例如，假设我想使用 `gpt-4o-mini` 来进一步了解人工智能。我可以问一个关于近期历史的问题，以及一个关于未来的问题：

In [ ]:
prompts = [
    "In a short sentence, has artifical intelligence grown in the last decade?",
    "In a short sentence, what are your thoughts on the future of artificial intelligence?",
]

for prompt in prompts:
    API_RESPONSE = get_completion(
        [{"role": "user", "content": prompt}],
        model="gpt-4o-mini",
        logprobs=True,
    )

    logprobs = [token.logprob for token in API_RESPONSE.choices[0].logprobs.content]
    response_text = API_RESPONSE.choices[0].message.content
    response_text_tokens = [token.token for token in API_RESPONSE.choices[0].logprobs.content]
    max_starter_length = max(len(s) for s in ["Prompt:", "Response:", "Tokens:", "Logprobs:", "Perplexity:"])
    max_token_length = max(len(s) for s in response_text_tokens)
    

    formatted_response_tokens = [s.rjust(max_token_length) for s in response_text_tokens]
    formatted_lps = [f"{lp:.2f}".rjust(max_token_length) for lp in logprobs]

    perplexity_score = np.exp(-np.mean(logprobs))
    print("Prompt:".ljust(max_starter_length), prompt)
    print("Response:".ljust(max_starter_length), response_text, "\n")
    print("Tokens:".ljust(max_starter_length), " ".join(formatted_response_tokens))
    print("Logprobs:".ljust(max_starter_length), " ".join(formatted_lps))
    print("Perplexity:".ljust(max_starter_length), perplexity_score, "\n")

Prompt:     In a short sentence, has artifical intelligence grown in the last decade?
Response:   Yes, artificial intelligence has grown significantly in the last decade, advancing in capabilities and applications across various fields. 

Tokens:                Yes              ,     artificial   intelligence            has          grown  significantly             in            the           last         decade              ,      advancing             in   capabilities            and   applications         across        various         fields              .
Logprobs:            -0.00           0.00          -0.00           0.00          -0.00          -0.73          -0.00          -0.01          -0.02          -0.00           0.00          -0.02          -0.66          -0.03          -0.62          -0.47          -0.02          -0.39          -0.01          -0.20          -0.00
Perplexity: 1.1644170003987546 

Prompt:     In a short sentence, what are your thoughts on the future of a

在这个示例中，对于关于近期历史、更具确定性的问题，`gpt-4o-mini` 返回了较低的困惑度分数；而对于关于近期未来、更具推测性的评估，则返回了较高的困惑度分数。同样，虽然这些差异并不能保证准确性，但它们有助于为我们解读模型结果以及今后如何使用这些结果指明方向。

## 6. 结论

很好！我们使用 `logprobs` 参数构建了一个更稳健的分类器，评估了问答系统的检索效果，并对 token 的每个“字节”进行了编码和解码！`logprobs` 为补全输出增添了有用的信息和信号，我们很期待看到开发者如何利用它来改进应用。

## 7. 可能的扩展

`logprobs` 还有许多本 cookbook 未涉及的其他用例。我们可以将 `logprobs` 用于：
  - 内容审核
  - 关键词选择
  - 改进提示以及输出的可解释性
  - Token 修复（token healing）
  - 以及更多！